# Aflossingsplannen voor studieleningen vergelijken met PROC LOAN

## Samenvatting

Een bureau voor studiefinanciering beoordeelt hoe een lichting afstuderenden een representatief saldo van $27,500 het beste kan aflossen door vier aflossingsstructuren te vergelijken — een federaal vastrentend standaardplan, een particuliere herfinanciering met afsluitprovisie, een variabele lening met rentetop (ARM) en een door de werkgever gesponsorde renteverlaging — met behulp van `PROC LOAN`. Over een looptijd van 120 maanden komen de vier aanbiedingen bij hun opgegeven startrente netjes overeen qua maandelijkse betaling en totale rente: het federale standaardplan kost de meeste rente (**$10,021.22** tegen 6.53%, betaling **$312.68**), de particuliere herfinanciering verlaagt de rente maar voegt $412.50 kosten toe (**$9,120.20** tegen 5.99%, betaling **$305.17**), en de lager genoteerde ARM (**$7,099.76** tegen 4.75%, betaling **$288.33**) en de werkgeversrenteverlaging (**$6,700.67** tegen 4.50%, betaling **$285.01**) hebben de laagste rentelasten. Een `COMPARE`-blok rapporteert vervolgens per plan de cumulatieve rente, hoofdsom en het openstaande saldo op 36, 60 en 120 maanden, en laat zien hoe ver elke lening is afgelost op de momenten waarop een lener het meest waarschijnlijk herfinanciert of aflost.

## Gegevensbronnen

| Dataset | Rijen | Beschrijving | Belangrijkste variabelen |
|---------|------|-------------|---------------|
| `borrowers` | 40 | Synthetische leningprofielen van een lichting afstuderenden, inline gegenereerd met `call streaminit(20260531)` en `rand('uniform')`. Gebruikt om realistische leenvoorwaarden voor de vergelijking te motiveren. | `student_id` (1001-1040), `balance` (hoofdsom bij afstuderen; deze trekking loopt van $11,800-$47,750, gemiddeld $30,771), `apr` (4.10%-9.10% jaarlijkse nominale rente, gemiddeld 6.50%), `term` (120 of 180 maanden, gemiddeld 144), `origination_pct` (1.0%-2.0% kosten, gemiddeld 1.50%) |

De representatieve lener die met `PROC LOAN` wordt geanalyseerd (bedrag $27,500, looptijd 120 maanden, start juli 2026) bevindt zich in de onderste middenmoot van de saldo- en renteverdeling van deze cohort; er worden geen externe gegevens of netwerkgegevens gebruikt. De cohort bestaat om plausibele leenvoorwaarden te motiveren — de directe vergelijking wordt uitgevoerd op de ene representatieve lening.

# Aflossingsplannen voor studieleningen vergelijken met PROC LOAN

Wanneer studenten afstuderen, moet een bureau voor studiefinanciering hen helpen kiezen tussen concurrerende aflossingsaanbiedingen. `PROC LOAN` (SAS/ETS) is hier precies voor gebouwd: het lost leningen met vaste rente, variabele rente (ARM) en renteverlaging (buydown) af en vergelijkt ze vervolgens naast elkaar op betaling, totale rente en aflossingsvoortgang.

In dit notebook doen we het volgende:

1. We genereren een synthetische lichting afstuderenden om realistische leenvoorwaarden vast te stellen.
2. We vatten de cohort samen met `PROC MEANS`.
3. We bouwen vier aflossingsplannen voor een representatief saldo van $27,500 en vergelijken ze met `PROC LOAN` (FIXED / ARM / BUYDOWN + COMPARE).
4. We voeren het aanbevolen vastrentende plan opnieuw afzonderlijk uit om de zelfstandige economie ervan te bevestigen.

Alles draait lokaal op inline gegenereerde synthetische gegevens — geen externe bestanden of netwerktoegang.

## Stap 1 — Genereer een synthetische lichting afstuderenden

We simuleren 40 leners. Elke lener krijgt een hoofdsom bij afstuderen, een JKP die losjes gekoppeld is aan de kredietwaardigheid, een standaard aflossingstermijn (10 of 15 jaar) en een afsluitprovisiepercentage. De seed maakt de gegevens reproduceerbaar.

In [1]:
GEGEVENS borrowers;
   CALL streaminit(20260531);
   LENGTE plan $ 28;
   DOE student_id = 1001 TOT 1040;
      /* Hoofdsom bij afstuderen: $9,500 - $48,000 */
      balance = round(9500 + 38500*rand('uniform'), 50);
      /* Jaarlijkse nominale JKP per kredietklasse: 4.0% - 9.5% */
      apr = round(4.0 + 5.5*rand('uniform'), 0.05);
      /* Standaard aflossingstermijn: 120 of 180 maanden */
      ALS rand('uniform') < 0.6 DAN term = 120;
      ANDERS term = 180;
      /* Afsluitprovisie als percentage van de hoofdsom: 1.0% - 2.0% */
      origination_pct = round(1.0 + rand('uniform'), 0.10);
      UITVOER;
   EINDE;
UITVOEREN;



NOTE: DATA borrowers


NOTE: Wrote borrowers (40 rows, 6 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds


## Stap 2 — Profileer de cohort

Voordat we individuele leningen modelleren, bekijken we de verdeling van saldi, rentes en looptijden. Dit laat zien hoe een *representatieve* lener eruitziet — de basis voor de directe vergelijking die volgt.

In [2]:
PROCEDURE GEMIDDELDEN GEGEVENS=borrowers n mean MIN MAX maxdec=2;
   label balance='Saldo ($)' apr='JKP (%)' term='Looptijd (maanden)' origination_pct='Afsluitprovisie (%)';
   VARIABELE balance apr term origination_pct;
UITVOEREN;


                                                  The MEANS Procedure

 Variable         Label                      N           Mean     Minimum     Maximum
 ------------------------------------------------------------------------------------
 balance          Saldo ($)                 40       30771.25    11800.00    47750.00
 apr              JKP (%)                   40           6.50        4.10        9.10
 term             Looptijd (maanden)        40         144.00      120.00      180.00
 origination_pct  Afsluitprovisie (%)       40           1.50        1.00        2.00
 ------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Stap 3 — Vergelijk vier aflossingsplannen voor een representatieve lener

Met een representatief saldo van $27,500 over een looptijd van 120 maanden (10 jaar), startend in juli 2026, zetten we vier realistische aanbiedingen naast elkaar:

- **Federale standaardlening (niet gesubsidieerd)** — een eenvoudige vastrentende lening tegen 6.53%.
- **Particuliere herfinanciering (met kosten)** — een lagere vaste rente van 5.99%, maar met $412.50 initialisatiekosten (`INIT=`).
- **Particuliere variabele lening (ARM)** — een variabele lening van 4.75% met een `CAPS=` van 1% per aanpassing / 5% over de looptijd, een `MAXRATE=` van 9.75%, een jaarlijkse `ADJUSTFREQ=` en het sleutelwoord `WORSTCASE`.
- **Werkgever 2-1 renteverlaging** — een gesubsidieerde start van 4.50% die, volgens het opgegeven schema, via `BDRATES=` stijgt naar 5.50% in jaar 2 en 6.50% daarna.

De `COMPARE`-instructie vraagt om het vergelijkende overzicht op 36, 60 en 120 maanden, met een `TAXRATE=` van 22% en een minimaal aantrekkelijk rendement van 4% (`MARR=`); `OUTSUM=` en `OUTCOMP=` leggen de samenvatting per lening en de vergelijkingsrijen vast. De onderstaande listing rapporteert voor elk plan en elk controlepunt de **cumulatieve betaalde rente, de cumulatief afgeloste hoofdsom en het openstaande saldo** — een duidelijk beeld van de aflossingsvoortgang tussen de kandidaten.

> **Opmerking over renteaanpassingen.** De `PROC LOAN` van Jenner lost elk plan af tegen de opgegeven **start**rente, dus de instellingen `CAPS`/`MAXRATE`/`WORSTCASE` van de ARM en de `BDRATES`-stappen van de renteverlaging worden in de listing weergegeven als leenvoorwaarden, maar worden **niet** toegepast op de betalingsberekening — de onderstaande cijfers voor de ARM en de renteverlaging weerspiegelen hun startrentes van 4.75% en 4.50%, constant gehouden over de volledige looptijd. Beschouw deze twee totalen als beste-geval (startrente-)kosten, niet als worstcase-plafonds.

In [3]:
PROCEDURE loan START=2026:7 amount=27500 life=120 outsum=plan_sum;

   fixed rate=6.53
         label='Federale standaardlening (niet gesubsidieerd)';

   fixed rate=5.99 init=412.50
         label='Particuliere herfinanciering (met kosten)';

   arm rate=4.75 caps=(1 5) maxrate=9.75 adjustfreq=12 worstcase
       label='Particuliere variabele lening (ARM, worstcase)';

   buydown rate=4.50 bdrates=(13=5.50 25=6.50)
           label='Werkgever 2-1 renteverlaging, daarna 6.5%';

   COMPARE at=(36 60 120) ALL taxrate=22 marr=4 OUTCOMP=plan_cmp;
UITVOEREN;


                            The LOAN Procedure

  Number of loans evaluated:    4

  Loan #1: Federale standaardlening (niet gesubsidieerd)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan #2: Particuliere herfinanciering (met kosten)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            5.9900
    Life (months):                          120
    Monthly Payment:                     305.17
    Total Paid (P&I):                  36620.20
    Total Interest:                     9120.20
    Initialization Cost:                 412.50

  Loan #3: Particuliere variabele lening (ARM, worstcase)
    Loan Type:                   Adjustable R


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Stap 4 — Voer het aanbevolen vastrentende plan opnieuw afzonderlijk uit

Voor de lener die waarde hecht aan betalingszekerheid, is het federale vastrentende standaardplan de veilige standaardkeuze. We voeren het opnieuw geïsoleerd uit om de belangrijkste economische kerncijfers te bevestigen: dezelfde maandelijkse betaling van **$312.68**, **$37,521.22** totaal betaald en **$10,021.22** totale rente die we zagen in de viervoudige vergelijking, nu gepresenteerd als een zelfstandige leningsamenvatting.

In [4]:
PROCEDURE loan START=2026:7 amount=27500 rate=6.53 life=120 schedule=yearly;
   fixed label='Federale standaardlening (niet gesubsidieerd)';
UITVOEREN;


                            The LOAN Procedure

  Number of loans evaluated:    1

  Loan #1: Federale standaardlening (niet gesubsidieerd)
    Loan Type:                   Fixed
    Amount (Principal):                27500.00
    Interest Rate (annual %):            6.5300
    Life (months):                          120
    Monthly Payment:                     312.68
    Total Paid (P&I):                  37521.22
    Total Interest:                    10021.22

  Loan Repayment Schedule: Federale standaardlening (niet gesubsidieerd)
      Year    Begin Balance        Payment       Interest      Principal      End Balance
    ------ ---------------- -------------- -------------- -------------- ----------------
         1         27500.00        3752.12        1736.12        2016.00         25484.00
         2         25484.00        3752.12        1600.47        2151.66         23332.34
         3         23332.34        3752.12        1455.68        2296.44         21035.90
         


NOTE: PROC LOAN loan analysis

NOTE: PROC LOAN statement used.


## Interpretatie van de resultaten

Alle vier de plannen lossen dezelfde hoofdsom van $27,500 volledig af over 120 maanden (het openstaande saldo van elk plan bereikt $0.00 in maand 120 in de COMPARE-tabel), dus de beslissing draait om de **maandelijkse betaling** en de **totale rente tegen de opgegeven rente**:

| Plan | Rente | Betaling | Totale rente | Opmerkingen |
|------|------|---------|----------------|-------|
| Federale standaardlening | 6.53% | $312.68 | $10,021.22 | Geen kosten; sterkste bescherming voor de lener |
| Particuliere herfinanciering | 5.99% | $305.17 | $9,120.20 | $412.50 afsluitprovisie |
| Particuliere variabele lening (ARM) | 4.75% (start) | $288.33 | $7,099.76 | Rente kan stijgen; cijfer is de kosten tegen startrente |
| Werkgever 2-1 renteverlaging | 4.50% (start) | $285.01 | $6,700.67 | Afhankelijk van voortgezet dienstverband |

- Het **federale standaard**plan is het duurst qua rente ($10,021.22), maar biedt een vaste, voorspelbare betaling van $312.68 zonder kosten.
- De **particuliere herfinanciering** verlaagt de rente naar 5.99% ($9,120.20 rente, $901 minder dan het federale plan), maar rekent $412.50 afsluitprovisie, dus het netto voordeel ten opzichte van het federale plan is ongeveer $488 aan rente minus kosten — alleen relevant als de lening lang genoeg loopt om niet te worden weggeherfinancierd.
- De **ARM** en de **renteverlaging** laten hier de laagste rente zien ($7,099.76 en $6,700.67), puur omdat hun **start**rentes ver onder de vastrentende aanbiedingen liggen. Zoals vermeld in Stap 3 houdt Jenner die startrentes constant, dus dit zijn beste-geval-cijfers: een echte ARM die naar boven bijstelt — of een renteverlaging die de werkgeverssubsidie verliest — zou hoger uitkomen, en de betaling is niet gegarandeerd.

De `COMPARE`-tabel verscherpt dit door te laten zien hoe snel elk plan aflost. Na 36 maanden heeft het federale plan $4,792.27 rente betaald en $6,464.10 hoofdsom afgelost (saldo $21,035.90), terwijl de renteverlaging slechts $3,263.97 rente heeft betaald en $6,996.24 hoofdsom heeft afgelost (saldo $20,503.76) — de plannen met lagere rente kosten zowel minder rente als lossen sneller af in de eerste drie jaar. Voor een risicomijdende afgestudeerde rechtvaardigt de rentezekerheid van het federale standaardplan vaak de hogere rente; voor een lener die vertrouwen heeft in een stabiel, langdurig dienstverband levert de gesubsidieerde start van de renteverlaging de grootste besparing op onder de opties die hun rente daadwerkelijk vastzetten.